# 08 - Baseline Routing Arm: TF-IDF + Linear SVM (all-in-notebook)

Self-contained and PEP 8 compliant. Reads the nested JSONL from notebook 07,
trains a class-weighted TF-IDF + Linear SVM, tunes an abstain (route-to-human)
threshold on validation, and evaluates on the held-out test set with macro-F1
and a per-class confusion matrix.


## 1. Setup, imports, and data location

In [1]:
try:
    import google.colab  # noqa: F401
    !pip -q install scikit-learn pandas
except Exception:
    pass

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Toggle: fold nested instruction text into the classifier features.
USE_INSTRUCTIONS = False


def find_ml_dir():
    """Locate data/ml (Colab Drive or a local path)."""
    candidates = [
        Path("/content/drive/MyDrive/newstart_ai"),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    for root in candidates:
        if (root / "data" / "ml" / "train.jsonl").exists():
            return root / "data" / "ml"
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        return Path("/content/drive/MyDrive/newstart_ai/data/ml")
    except Exception as exc:
        raise FileNotFoundError(
            "Run notebook 07 first to create data/ml/*.jsonl"
        ) from exc


ML_DIR = find_ml_dir()
REPORTS = ML_DIR.parent.parent / "reports"
MODELS = ML_DIR.parent.parent / "models"
REPORTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

label_map = json.loads((ML_DIR / "label_map.json").read_text())
id2label = {value: key for key, value in label_map.items()}
labels_sorted = [id2label[i] for i in range(len(label_map))]
class_weights = json.loads(
    (ML_DIR / "class_weights.json").read_text()
)["by_id"]
print("labels:", labels_sorted)

labels: ['DMV', 'IRS', 'SSA', 'USCIS']


## 2. Load the nested JSONL splits

In [3]:
def build_text(row):
    """Form text, optionally with nested instruction text appended."""
    text = str(row["text"])
    if USE_INSTRUCTIONS:
        associated = row.get("associated_instructions")
        if isinstance(associated, list):
            extra = "\n".join(
                item.get("text", "") for item in associated
            )
            text = text + "\n" + extra
    return text


def load_split(name):
    """Load one JSONL split and build the model_text column."""
    frame = pd.read_json(ML_DIR / f"{name}.jsonl", lines=True)
    frame["model_text"] = frame.apply(build_text, axis=1)
    return frame


train = load_split("train")
val = load_split("val")
test = load_split("test")
print("loaded:", len(train), len(val), len(test))
print("USE_INSTRUCTIONS =", USE_INSTRUCTIONS)

loaded: 416 89 82
USE_INSTRUCTIONS = False


## 3. Shared evaluation helpers (macro-F1, confusion)

In [4]:
def evaluate(y_true, y_pred, arm, proba=None, threshold=0.0):
    """Score predictions: macro-F1, per-class, confusion, coverage."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ids = list(range(len(labels_sorted)))
    mask = np.ones(len(y_true), dtype=bool)
    if proba is not None and threshold > 0:
        mask = proba.max(axis=1) >= threshold
    true_kept = y_true[mask]
    pred_kept = y_pred[mask]
    report = classification_report(
        true_kept,
        pred_kept,
        labels=ids,
        target_names=labels_sorted,
        output_dict=True,
        zero_division=0,
    )
    macro_f1 = f1_score(
        true_kept, pred_kept, labels=ids,
        average="macro", zero_division=0,
    )
    accuracy = accuracy_score(true_kept, pred_kept)
    matrix = confusion_matrix(true_kept, pred_kept, labels=ids)
    return {
        "arm": arm,
        "n_test": int(len(y_true)),
        "n_scored": int(mask.sum()),
        "coverage": round(float(mask.mean()), 4),
        "accuracy": round(float(accuracy), 4),
        "macro_f1": round(float(macro_f1), 4),
        "per_class_f1": {
            name: round(report[name]["f1-score"], 4)
            for name in labels_sorted
        },
        "per_class_recall": {
            name: round(report[name]["recall"], 4)
            for name in labels_sorted
        },
        "confusion_matrix": matrix.tolist(),
        "labels": labels_sorted,
    }


def print_confusion(metrics):
    """Print the confusion matrix (rows = true, cols = pred)."""
    width = max(len(name) for name in labels_sorted) + 1
    header = " " * (width + 1) + " ".join(
        f"{name[:6]:>6}" for name in labels_sorted
    )
    print(header)
    for i, row in enumerate(metrics["confusion_matrix"]):
        body = " ".join(f"{value:>6}" for value in row)
        print(f"{labels_sorted[i]:<{width}} {body}")

## 4. Train and tune the abstain threshold

``LinearSVC`` has no probabilities, so we wrap it in ``CalibratedClassifierCV``
to obtain confidences for the abstain rule. Class weights are ``balanced`` so the
rare IRS class is not ignored.


In [5]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
        strip_accents="unicode",
    )),
    ("clf", CalibratedClassifierCV(
        LinearSVC(class_weight="balanced"),
        method="sigmoid",
        cv=3,
    )),
])
pipeline.fit(train["model_text"], train["label"])

val_conf = pipeline.predict_proba(val["model_text"]).max(axis=1)
threshold = 0.0
for candidate in np.linspace(0, 0.95, 20):
    if (val_conf >= candidate).mean() >= 0.85:
        threshold = round(float(candidate), 3)
print("tuned abstain threshold:", threshold)

tuned abstain threshold: 0.75


## 5. Evaluate on the test set

In [6]:
test_proba = pipeline.predict_proba(test["model_text"])
y_pred = test_proba.argmax(axis=1)
metrics = evaluate(
    test["label"].values,
    y_pred,
    arm="baseline_tfidf_svm",
    proba=test_proba,
    threshold=threshold,
)
metrics["abstain_threshold"] = threshold
print("macro-F1:", metrics["macro_f1"])
print("accuracy:", metrics["accuracy"])
print("coverage:", metrics["coverage"])
print("per-class recall:", metrics["per_class_recall"])
print()
print("Confusion (rows=true, cols=pred):")
print_confusion(metrics)

macro-F1: 1.0
accuracy: 1.0
coverage: 0.9634
per-class recall: {'DMV': 1.0, 'IRS': 1.0, 'SSA': 1.0, 'USCIS': 1.0}

Confusion (rows=true, cols=pred):
          DMV    IRS    SSA  USCIS
DMV        42      0      0      0
IRS         0      1      0      0
SSA         0      0     22      0
USCIS       0      0      0     14


## 6. Save metrics

In [7]:
metrics_path = REPORTS / "metrics_baseline_tfidf_svm.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print("saved ->", metrics_path)

saved -> /sessions/magical-nice-archimedes/mnt/FinalSchoolProject/newstart-ai-dev-petra/newstart-ai-dev-petra/newstart_ai_jeffi/reports/metrics_baseline_tfidf_svm.json


## 7. Reading the result

On full form text the agency is nearly linearly separable, so this baseline is
very strong. The honest signals are the IRS confusion row and behaviour under
harder inputs. Set ``USE_INSTRUCTIONS = True`` in cell 1 to fold the nested
instruction text into the features and compare.
